<a href="https://colab.research.google.com/github/RajeshworM/Yield_Modelling_Automation/blob/main/Basmati_Export_Countrywise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# INSTALL SYSTEM DEPENDENCIES
# =========================================================

!apt-get update -qq

!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpango-1.0-0 \
    libcairo2 \
    libnss3 \
    libxshmfence1 \
    libglu1-mesa

# =========================================================
# INSTALL PYTHON PACKAGES
# =========================================================

!pip -q install playwright pandas lxml beautifulsoup4 tqdm openpyxl nest_asyncio

# =========================================================
# INSTALL PLAYWRIGHT BROWSER
# =========================================================

!playwright install chromium

# =========================================================
# IMPORTS
# =========================================================

import nest_asyncio
nest_asyncio.apply()

import pandas as pd

from tqdm import tqdm
from playwright.async_api import async_playwright

# =========================================================
# SETTINGS
# =========================================================

URL = "https://tradestat.commerce.gov.in/ftspcc/export_commodity_wise_all_countries"

START_YEAR = 2018

MONTHS = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

VALUE_TYPES = [
    "Quantity",
    "US $ Millions",
    "₹ Crore",
    "₹/Unit",
    "US$/ Unit"
]

all_data = []

# =========================================================
# SCRAPER
# =========================================================

async def scrape():

    global all_data

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox"]
        )

        page = await browser.new_page()

        print("✅ Browser Started")

        # =================================================
        # OPEN WEBSITE
        # =================================================

        await page.goto(URL, timeout=120000)

        await page.wait_for_timeout(5000)

        print("✅ Website Loaded")

        # =================================================
        # GET YEARS
        # =================================================

        year_options = await page.locator("select").nth(1).locator("option").all_text_contents()

        available_years = []

        for y in year_options:

            y = y.strip()

            if y.isdigit():

                yr = int(y)

                if yr >= START_YEAR:
                    available_years.append(yr)

        available_years = sorted(list(set(available_years)))

        print("✅ Years:", available_years)

        # =================================================
        # MAIN LOOP
        # =================================================

        for year in tqdm(available_years, desc="Years"):

            for month in MONTHS:

                for value_type in VALUE_TYPES:

                    try:

                        # -------------------------------------
                        # RELOAD PAGE
                        # -------------------------------------

                        await page.goto(URL, timeout=120000)

                        await page.wait_for_timeout(3000)

                        selects = page.locator("select")

                        # -------------------------------------
                        # MONTH
                        # -------------------------------------

                        await selects.nth(0).select_option(label=month)

                        # -------------------------------------
                        # YEAR
                        # -------------------------------------

                        await selects.nth(1).select_option(label=str(year))

                        # -------------------------------------
                        # COMMODITY
                        # -------------------------------------

                        commodity_options = await selects.nth(2).locator("option").all_text_contents()

                        commodity_found = False

                        for option in commodity_options:

                            txt = option.strip().upper()

                            if "RICE -BASMOTI" in txt or "RICE-BASMOTI" in txt:

                                await selects.nth(2).select_option(label=option.strip())

                                commodity_found = True

                                break

                        if not commodity_found:

                            print("❌ Commodity Not Found")

                            continue

                        # -------------------------------------
                        # VALUE TYPE
                        # -------------------------------------

                        await selects.nth(3).select_option(label=value_type)

                        # -------------------------------------
                        # SUBMIT
                        # -------------------------------------

                        await page.locator("button:has-text('Submit')").click()

                        await page.wait_for_timeout(5000)

                        # -------------------------------------
                        # READ TABLE
                        # -------------------------------------

                        html = await page.content()

                        tables = pd.read_html(html)

                        if len(tables) == 0:

                            print(f"❌ No Table -> {month} {year} {value_type}")

                            continue

                        df = tables[0]

                        # -------------------------------------
                        # ADD META
                        # -------------------------------------

                        df["Selected_Month"] = month
                        df["Selected_Year"] = year
                        df["Value_Type"] = value_type

                        all_data.append(df)

                        print(f"✅ DONE -> {month} | {year} | {value_type}")

                    except Exception as e:

                        print(f"\n❌ ERROR -> {month} | {year} | {value_type}")

                        print(e)

        await browser.close()

# =========================================================
# RUN
# =========================================================

await scrape()

# =========================================================
# SAVE CSV
# =========================================================

if len(all_data) > 0:

    final_df = pd.concat(all_data, ignore_index=True)

    final_df.columns = [
        str(c).replace("\n", " ").strip()
        for c in final_df.columns
    ]

    final_df = final_df[
        final_df.iloc[:,0] != "S.No."
    ]

    csv_file = "Rice_Basmoti_Countrywise_2018_Latest.csv"

    final_df.to_csv(csv_file, index=False)

    print("\n===================================")
    print("✅ CSV SAVED")
    print("===================================")

    print(csv_file)

    from google.colab import files
    files.download(csv_file)

else:

    print("❌ No data scraped")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libxdamage1 is already the newest version (1:1.1.5-2build2).
libxdamage1 set to manually installed.
libxfixes3 is already the newest version (1:6.0.0-1).
libxfixes3 set to manually installed.
libxkbcommon0 is already the newest version (1.4.0-1).
libxkbcommon0 set to manually installed.
libxrandr2 is already the newest version (2:1.5.2-1build1).
libxrandr2 set to manually installed.
libxshmfence1 is already the newest version (1.3-1build4).
libxshmfence1 set to manually installed.
libasound2 is already the newest version (1.2.6.1-1ubuntu1.1).
libasound2 set to manually installed.
libcairo2 is already the newest version (1.16.0-5ubuntu2.1).
libcairo2 set to manually installed.
libcups2 is already the newes

Years:   0%|          | 0/9 [00:00<?, ?it/s]/tmp/ipykernel_2542/1689200114.py:189: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)



❌ ERROR -> January | 2018 | Quantity
No tables found

❌ ERROR -> January | 2018 | US $ Millions
Locator.select_option: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("select").nth(3)
    - locator resolved to <select id="ReportValAce" class="form-select" name="ReportValAce" wire:model="ReportValAce">…</select>
  - attempting select option action
    2 × waiting for element to be visible and enabled
      - did not find some options
    - retrying select option action
    - waiting 20ms
    2 × waiting for element to be visible and enabled
      - did not find some options
    - retrying select option action
      - waiting 100ms
    60 × waiting for element to be visible and enabled
       - did not find some options
     - retrying select option action
       - waiting 500ms

✅ DONE -> January | 2018 | ₹ Crore


/tmp/ipykernel_2542/1689200114.py:189: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


✅ DONE -> January | 2018 | ₹/Unit


...

✅ DONE -> December | 2026 | ₹/Unit


/tmp/ipykernel_2542/1689200114.py:189: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)
Years: 100%|██████████| 9/9 [2:21:55<00:00, 946.15s/it]


✅ DONE -> December | 2026 | US$/ Unit

✅ CSV SAVED
Rice_Basmoti_Countrywise_2018_Latest.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>